# segment-line-intersect-2d — ex1: intersect a segment with an infinite line in 2-D

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `segment-line-intersect-2d`. Running the final beacon cell reports progress against the `Geometry: Segment-line intersect 2-D` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Segment-line intersect 2-D` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`segment-line-intersect-2d`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "segment-line-intersect-2d"
DD_SUBTOPIC = "Geometry: Segment-line intersect 2-D"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## 2-D segment vs line intersection — quick refresher

**Parametric form.** A segment from `P` to `P + d` (direction = endpoint - start) has points `P + t*d` for `t ∈ [0, 1]`. A line through `Q` with direction `e` has points `Q + s*e` for `s ∈ ℝ` (no bound).

**Intersection.** Solve `P + t*d = Q + s*e`, i.e. `t*d - s*e = Q - P`. In matrix form, stack `d` and `-e` as columns:
```
[d_x  -e_x] [t]   [Q_x - P_x]
[d_y  -e_y] [s] = [Q_y - P_y]
```
Solve via Cramer's rule (or `torch.linalg.solve`). If the determinant is zero, segment and line are parallel — no unique intersection.

**Hit test.** The segment hits the line iff a solution exists AND `0 ≤ t ≤ 1`. `s` is unconstrained (the line is infinite). Returning `(t, s, hit_bool)` is the canonical signature.

### Exercise 1 — intersect a segment with an infinite line in 2-D

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the parametric-form linear system (with `torch.linalg.solve`) to find segment-vs-line intersection in 2-D and return whether the segment actually crosses the line.
> Keywords: 2D-geometry, cramer, linalg-solve, parametric
> ```

**KCs targeted:** `segment-line-as-linear-system`, `t-in-zero-one-hit-test`

Implement `ex1_seg_line_intersect(P, Q1, Q2, R1, R2)`. Inputs are all 2-D points:
- `P`, `Q1, Q2`: the SEGMENT goes from `Q1` to `Q2`; ignore `P` (legacy name compat).
  Actually let's simplify: drop unused names.

Let's use clean names. Inputs:
- `S0`, `S1`: 2-D endpoints of the segment.
- `L0`, `L1`: 2-D points defining the infinite line.

Steps:
1. Direction of segment: `d = S1 - S0`. Direction of line: `e = L1 - L0`.
2. Solve `t*d - s*e = L0 - S0` as a 2×2 linear system:
   ```python
   A = t.stack([d, -e], dim=1)  # shape (2, 2)
   b = L0 - S0                  # shape (2,)
   ts = t.linalg.solve(A, b)    # ts = [t, s]
   ```
3. Compute `hit = (ts[0] >= 0) & (ts[0] <= 1)`. The segment parameter `t` must be in `[0, 1]`; line `s` is unconstrained.
4. Return `(ts[0].item(), ts[1].item(), bool(hit.item()))`.

Assume the matrix is non-singular for this drill (parallel case is covered separately in `try-except-solve`).

Inputs are `(2,)` float tensors. Output is `(t_seg, s_line, hit)`.

In [ ]:
def ex1_seg_line_intersect(S0: Tensor, S1: Tensor, L0: Tensor, L1: Tensor):
    d = S1 - S0
    e = L1 - L0
    A = t.stack([d, -e], dim=1)   # columns are d and -e
    b = L0 - S0
    ts = t.linalg.solve(A, b)
    t_seg, s_line = ts[0].item(), ts[1].item()
    hit = (t_seg >= 0.0) and (t_seg <= 1.0)
    return t_seg, s_line, hit


<details><summary>Solution</summary>

```python
def ex1_seg_line_intersect(S0: Tensor, S1: Tensor, L0: Tensor, L1: Tensor):
    d = S1 - S0
    e = L1 - L0
    A = t.stack([d, -e], dim=1)   # columns are d and -e
    b = L0 - S0
    ts = t.linalg.solve(A, b)
    t_seg, s_line = ts[0].item(), ts[1].item()
    hit = (t_seg >= 0.0) and (t_seg <= 1.0)
    return t_seg, s_line, hit
```

**Why stack `[d, -e]` not `[d, e]`.** The equation is `P + t*d = Q + s*e`, rearranged to `t*d - s*e = Q - P`. Multiplying through the system, the second column is `-e`. Get this wrong and `s` flips sign — easy bug to miss because `t` stays correct.

**Why `[0, 1]` not `(0, 1)`.** Closed interval — endpoints on the line count as hits. Half-open conventions (`[0, 1)`) appear in raycasting (next-segment continuation) but for the canonical intersection test, both endpoints are part of the segment.

**This generalizes to 3-D.** Replace the 2×2 with a 3×2 system (under-determined — line and segment in 3-D *usually* miss). For 3-D you'd use `torch.linalg.lstsq` and check residual to decide whether 'close enough' is a hit. Different drill.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()